# 01 — Exploratory Data Analysis

Goals:
1. Verify the dataset loads correctly (real or synthetic fallback)
2. Check the engineered fraud label rate
3. Identify univariate fraud signals
4. Spot data quality issues we need to handle in the feature pipeline

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import LendingClubLoader, build_fraud_labels, LabelConfig

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

In [ ]:
# Load — falls back to synthetic if real data isn't downloaded yet
loader = LendingClubLoader(
    raw_path='../data/raw/accepted_2007_to_2018Q4.csv',
    sample_size=50_000,
    random_seed=42,
)
df = loader.load()
df = build_fraud_labels(df, LabelConfig())
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# Label rate & rule-level breakdown
print(f'Fraud rate: {df.is_fraud.mean():.2%}')
df[['rule_fpd', 'rule_income_anomaly', 'rule_debt_inconsist', 'rule_address_ring']].mean()

In [ ]:
# Fraud rate by grade — should increase monotonically
fraud_by_grade = df.groupby('grade')['is_fraud'].mean().sort_index()
fraud_by_grade.plot(kind='bar', figsize=(8, 4), title='Fraud rate by grade')
plt.ylabel('Fraud rate'); plt.show()

In [ ]:
# DTI distribution: fraud vs. clean
fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [(0, '#2E86AB'), (1, '#C73E1D')]:
    sub = df[df.is_fraud == label]['dti'].dropna()
    ax.hist(sub, bins=50, alpha=0.5, label=f'fraud={label}', color=color, density=True)
ax.set_xlabel('DTI'); ax.legend(); ax.set_title('DTI by label')
plt.show()

In [ ]:
# Missingness profile
miss = (df.isna().sum().sort_values(ascending=False) / len(df))
miss[miss > 0].head(20).plot(kind='barh', figsize=(8, 6), title='Top missing-value columns')
plt.xlabel('Missing fraction'); plt.show()

In [ ]:
# Verification status × fraud rate (key business slice)
df.groupby('verification_status')['is_fraud'].agg(['mean', 'count']).round(4)